# 04 - Context Window Management (上下文窗口管理)

## 学习目标
- 理解 token 预算的概念和分配策略
- 使用 tiktoken 精确计算 token 数量
- 实现滑动窗口和摘要压缩技术
- 构建分层记忆系统（短期/中期/长期）
- 掌握上下文溢出预防机制

In [ ]:
# 初始化环境
import os
import json
import math
import hashlib
from typing import Any
from dataclasses import dataclass, field
from collections import deque

# 尝试导入 tiktoken
try:
    import tiktoken
    TIKTOKEN_AVAILABLE = True
    print("tiktoken 已就绪")
except ImportError:
    TIKTOKEN_AVAILABLE = False
    print("⚠ tiktoken 未安装，将使用字符估算。安装: pip install tiktoken")

print("环境初始化完成")

---
## 1. Token 预算分配策略

### 预算分配表
```
+---------------------+-------+-----------------------+
| 组件                 | 占比   | 说明                   |
+---------------------+-------+-----------------------+
| System Prompt       | 15%   | 角色定义和基础规则      |
| Few-shot Examples   |  8%   | 2-5 个标注示例          |
| Conversation History | 50%   | 多轮对话历史            |
| Knowledge / RAG     | 15%   | 检索到的外部知识        |
| Output Allocation   | 12%   | 为模型输出预留空间      |
+---------------------+-------+-----------------------+
```

### 设计原理
- **History 占 50%**：多轮对话是最消耗 token 的组件，也是任务上下文的核心
- **Output 占 12%**：必须为输出预留空间，否则模型可能截断回答
- **System 仅占 15%**：过长的系统提示反而降低遵循度

In [ ]:
# ============================================================
# Token 计数引擎
# ============================================================

class TokenCounter:
    """
    Token 精确计数器：支持 tiktoken 和估算两种模式。
    
    What: 统一接口的 token 计数工具。
    Why: 精确的 token 计数是上下文管理的基础，不同模型使用不同的编码器。
    When: 任何需要管理上下文窗口的场景。
    """
    
    # 常见模型的编码器映射
    MODEL_ENCODING_MAP = {
        "gpt-4o": "o200k_base",
        "gpt-4o-mini": "o200k_base",
        "gpt-4": "cl100k_base",
        "gpt-4-turbo": "cl100k_base",
        "gpt-3.5-turbo": "cl100k_base",
        "text-embedding-3-small": "cl100k_base",
        "text-embedding-ada-002": "cl100k_base",
    }
    
    # 上下文窗口大小
    MODEL_MAX_TOKENS = {
        "gpt-4o": 128000,
        "gpt-4o-mini": 128000,
        "gpt-4": 8192,
        "gpt-4-turbo": 128000,
        "gpt-4-32k": 32768,
        "gpt-3.5-turbo": 4096,
        "gpt-3.5-turbo-16k": 16384,
    }
    
    def __init__(self, model: str = "gpt-4o"):
        self.model = model
        self._encoding = None
        if TIKTOKEN_AVAILABLE:
            encoding_name = self.MODEL_ENCODING_MAP.get(model, "cl100k_base")
            try:
                self._encoding = tiktoken.get_encoding(encoding_name)
            except Exception:
                self._encoding = tiktoken.get_encoding("cl100k_base")
    
    def count(self, text: str) -> int:
        """
        计算文本的 token 数。
        
        Args:
            text: 输入文本
        Returns:
            token 数量
        """
        if self._encoding and TIKTOKEN_AVAILABLE:
            return len(self._encoding.encode(text))
        else:
            # 估算：中文约 1.5 char/token，英文约 4 char/token
            chinese = sum(1 for c in text if '\u4e00' <= c <= '\u9fff')
            english = len(text) - chinese
            return int(chinese / 1.5 + english / 4.0)
    
    def count_messages(self, messages: list[dict]) -> int:
        """
        计算消息列表的 token 数。
        
        每条消息有固定开销（role + 格式标记），通常 4-6 tokens。
        """
        total = 0
        for msg in messages:
            total += 4  # 每条消息的格式开销
            for key, value in msg.items():
                if isinstance(value, str):
                    total += self.count(value)
                elif isinstance(value, list):
                    for item in value:
                        if isinstance(item, dict):
                            for v in item.values():
                                total += self.count(str(v))
        total += 2  # 回复的 priming token
        return total
    
    def get_max_tokens(self) -> int:
        """获取模型的最大上下文窗口大小"""
        return self.MODEL_MAX_TOKENS.get(self.model, 4096)

# 创建计数器
counter = TokenCounter("gpt-4o")

# 测试计数
test_texts = [
    "Hello, how are you?",
    "你好，今天天气怎么样？",
    "Artificial intelligence is transforming every industry worldwide.",
]

print("=== Token 计数测试 ===")
for text in test_texts:
    count = counter.count(text)
    print(f"'{text}' → {count} tokens")

# 测试消息列表计数
messages = [
    {"role": "system", "content": "你是一个有帮助的AI助手。请用JSON格式回答。"},
    {"role": "user", "content": "请分析以下文本的情感：这个产品非常好用！"},
    {"role": "assistant", "content": '{"sentiment": "positive", "confidence": 0.95}'},
]
msg_tokens = counter.count_messages(messages)
print(f"\n3 条消息: {msg_tokens} tokens")
print(f"模型最大上下文: {counter.get_max_tokens()} tokens")

In [ ]:
# ============================================================
# Token 预算管理器
# ============================================================

@dataclass
class BudgetReport:
    """Token 预算使用报告"""
    model: str
    max_tokens: int
    allocated: dict[str, int]   # 各组件分配的 token
    used: dict[str, int]        # 各组件实际使用的 token
    remaining: int               # 剩余 token
    warnings: list[str] = field(default_factory=list)

class TokenBudgetManager:
    """
    Token 预算管理器：按分配比例控制各组件的 token 使用。
    
    What: 基于预设的预算比例，跟踪和限制各组件的 token 消耗。
    Why: 防止某个组件（尤其是对话历史）消耗过多 token 导致溢出。
    When: 构建多轮对话系统或需要严格控制上下文窗口的任何场景。
    """
    
    # 默认预算分配比例
    DEFAULT_RATIOS = {
        "system": 0.15,        # System Prompt
        "fewshot": 0.08,       # Few-shot Examples
        "history": 0.50,       # Conversation History
        "knowledge": 0.15,     # Knowledge / RAG Context
        "output": 0.12,        # Output Allocation
    }
    
    def __init__(
        self,
        model: str = "gpt-4o",
        ratios: dict | None = None,
        safety_margin: float = 0.95,  # 安全边界：95% 时开始压缩
    ):
        self.counter = TokenCounter(model)
        self.ratios = ratios or self.DEFAULT_RATIOS
        self.max_tokens = self.counter.get_max_tokens()
        self.safety_margin = safety_margin
        self._used: dict[str, int] = {
            "system": 0, "fewshot": 0, "history": 0,
            "knowledge": 0, "output": 0
        }
    
    def get_allocation(self, component: str) -> int:
        """获取某组件的 token 预算上限"""
        ratio = self.ratios.get(component, 0.0)
        return int(self.max_tokens * ratio)
    
    def track(self, component: str, token_count: int):
        """记录某组件使用的 token 数"""
        self._used[component] = self._used.get(component, 0) + token_count
    
    def reset(self):
        """重置使用计数"""
        for key in self._used:
            self._used[key] = 0
    
    def check(self) -> BudgetReport:
        """
        生成当前预算使用报告。
        
        Returns:
            BudgetReport: 包含分配、使用、剩余和警告的完整报告
        """
        warnings = []
        allocated = {}
        for comp in self.ratios:
            alloc = self.get_allocation(comp)
            allocated[comp] = alloc
            used = self._used.get(comp, 0)
            if used > alloc:
                warnings.append(f"{comp}: 超出预算 {used - alloc} tokens ({used}/{alloc})")
        
        total_used = sum(self._used.values())
        remaining = self.max_tokens - total_used
        
        # 检查溢出风险
        usage_ratio = total_used / self.max_tokens
        if usage_ratio > self.safety_margin:
            warnings.append(f"总使用率 {usage_ratio:.1%} 超过安全边界 {self.safety_margin:.1%}")
        if remaining < 0:
            warnings.append(f"上下文溢出! 超出 {abs(remaining)} tokens")
        
        return BudgetReport(
            model=self.counter.model,
            max_tokens=self.max_tokens,
            allocated=allocated,
            used=dict(self._used),
            remaining=remaining,
            warnings=warnings,
        )

# 测试预算管理器
budget = TokenBudgetManager("gpt-4o")

print("=== Token 预算分配 ===")
for component, ratio in budget.ratios.items():
    alloc = budget.get_allocation(component)
    print(f"{component:12s}: {ratio:5.0%} → {alloc:>6,} tokens")

# 模拟使用
budget.track("system", 3000)
budget.track("fewshot", 1500)
budget.track("history", 50000)

report = budget.check()
print(f"\n=== 使用报告 ===")
print(f"模型: {report.model}")
print(f"使用: {report.used}")
print(f"剩余: {report.remaining:,} tokens")
if report.warnings:
    print(f"⚠ 警告: {report.warnings}")

---
## 2. 滑动窗口与重叠

### 原理
当对话历史超出 token 预算时，使用滑动窗口保留最近的对话轮次，同时保持部分重叠以避免上下文断裂。

In [ ]:
# ============================================================
# 滑动窗口实现
# ============================================================

@dataclass
class ConversationTurn:
    """对话轮次"""
    role: str  # "user" | "assistant" | "system"
    content: str
    tokens: int = 0
    turn_id: int = 0

class SlidingWindowManager:
    """
    滑动窗口对话管理器。
    
    What: 维护一个 token 受限的滑动窗口，自动截断旧消息。
    Why: 对话历史可能无限增长，必须在窗口满时智能丢弃最早的轮次。
    When: 长对话场景（客服、辅导、代码助手等）。
    """
    
    def __init__(
        self,
        max_tokens: int,
        overlap_turns: int = 2,  # 重叠轮次：新旧窗口之间保留的轮次
        model: str = "gpt-4o",
    ):
        self.max_tokens = max_tokens
        self.overlap_turns = overlap_turns
        self.counter = TokenCounter(model)
        self._turns: list[ConversationTurn] = []
        self._turn_counter = 0
        self._system_prompt: ConversationTurn | None = None
        self._total_tokens = 0
    
    def set_system_prompt(self, content: str):
        """设置系统提示（始终保留，不计入滑动窗口）"""
        tokens = self.counter.count(content)
        self._system_prompt = ConversationTurn(
            role="system", content=content, tokens=tokens, turn_id=0
        )
    
    def add_turn(self, role: str, content: str) -> bool:
        """
        添加对话轮次。如果超限，自动滑动窗口。
        
        Returns:
            True 如果添加成功，False 如果触发了截断
        """
        tokens = self.counter.count(content)
        turn = ConversationTurn(
            role=role, content=content, tokens=tokens,
            turn_id=self._turn_counter
        )
        self._turn_counter += 1
        
        self._turns.append(turn)
        self._total_tokens += tokens
        
        # 检查是否超限
        truncated = False
        while self._total_tokens > self.max_tokens and len(self._turns) > self.overlap_turns:
            removed = self._turns.pop(0)  # 移除最早的
            self._total_tokens -= removed.tokens
            truncated = True
        
        return not truncated
    
    def get_messages(self) -> list[dict]:
        """获取当前窗口内的消息列表（API 格式）"""
        messages = []
        if self._system_prompt:
            messages.append({
                "role": self._system_prompt.role,
                "content": self._system_prompt.content
            })
        for turn in self._turns:
            messages.append({"role": turn.role, "content": turn.content})
        return messages
    
    @property
    def token_usage(self) -> int:
        """当前窗口的总 token 数"""
        return self._total_tokens + (self._system_prompt.tokens if self._system_prompt else 0)
    
    @property
    def turn_count(self) -> int:
        """当前窗口内的轮次数量"""
        return len(self._turns)

# 演示滑动窗口
sw = SlidingWindowManager(max_tokens=500, overlap_turns=2, model="gpt-4o")
sw.set_system_prompt("你是一个智能客服助手。请用简洁的语言回答用户问题。")

print("=== 滑动窗口演示 ===")
conversation = [
    ("user", "你好，我想咨询一下退货政策。"),
    ("assistant", "您好！我们的退货政策是：购买后7天内可无理由退货，商品需保持完好。需要我帮您处理退货吗？"),
    ("user", "是的，我上周买了一件衣服，但是尺码不合适。"),
    ("assistant", "了解。请问您的订单号是多少？我帮您查询一下。"),
    ("user", "订单号是 ORD-2026-01589。"),
    ("assistant", "已查到您的订单。该订单在7天退货期内，可以办理退货。我会为您生成退货单号。"),
    ("user", "好的谢谢。退货地址是什么？"),
    ("assistant", "退货地址是：北京市朝阳区xxx路xxx号，退货仓。请将商品原包装寄回。"),
    ("user", "寄回去的运费谁承担？"),
    ("assistant", "尺码不合适属于个人原因退货，运费需要您自行承担。如果是质量问题，我们承担运费。"),
    ("user", "明白了，我今天就寄过去。大概多久能退款？"),
    ("assistant", "一般收到退货后3-5个工作日内退款到您的原支付账户。"),
]

for role, content in conversation:
    is_ok = sw.add_turn(role, content)
    status = "✓" if is_ok else "⚠ (触发截断)"
    print(f"  {status} [{role}] {content[:30]}... (窗口 tokens: {sw.token_usage}, 轮次: {sw.turn_count})")

print(f"\n最终窗口状态:")
print(f"  Token 使用: {sw.token_usage}")
print(f"  保留轮次: {sw.turn_count}/{len(conversation)}")
print(f"  最早保留: turn_id={sw._turns[0].turn_id}" if sw._turns else "  无消息")

---
## 3. 摘要压缩 (Summarization-Based Compression)

对于超出窗口的对话历史，使用 LLM 生成摘要作为压缩表示，释放 token 空间。

In [ ]:
# ============================================================
# 对话摘要压缩器
# ============================================================

@dataclass
class ConversationSummary:
    """对话摘要"""
    turn_range: tuple[int, int]  # 摘要覆盖的轮次范围
    summary_text: str
    token_count: int
    key_points: list[str]         # 关键信息点
    timestamp: str                 # ISO 时间戳

class ConversationCompressor:
    """
    对话摘要压缩器：将长对话历史压缩为摘要。
    
    What: 使用 LLM 将旧的对话轮次压缩为简洁的摘要。
    Why: 保留对话语境的语义信息，同时大幅减少 token 消耗（压缩比通常 5-20x）。
    When: 对话轮次超过 20 轮，或 token 使用超过窗口的 70% 时。
    """
    
    COMPRESSION_PROMPT = """请将以下对话历史总结为一个紧凑的摘要。
保留关键信息：用户的需求、给出的回答、达成的结论、待处理的事项。
不要包含客套话，只保留事实和决策。

对话历史：
{conversation}

摘要（用中文，不超过100字）：
关键信息点（每点一行，以 "- " 开头）："""
    
    def __init__(self, counter: TokenCounter | None = None):
        self.counter = counter or TokenCounter()
        self.summaries: list[ConversationSummary] = []
    
    def compress(
        self,
        turns: list[ConversationTurn],
        start_turn_id: int,
    ) -> ConversationSummary:
        """
        压缩一批对话轮次为摘要。
        
        注意：此方法模拟 LLM 调用。实际使用时需要真实的 LLM API。
        
        Args:
            turns: 要压缩的对话轮次列表
            start_turn_id: 起始轮次 ID
        Returns:
            ConversationSummary 对象
        """
        # 构建对话文本
        conv_text = "\n".join(
            f"[{turn.role}]: {turn.content}"
            for turn in turns
        )
        
        # 模拟摘要生成（实际应调用 LLM）
        # 此处使用启发式方法：提取关键短语
        summary = self._heuristic_summarize(turns)
        
        end_turn_id = start_turn_id + len(turns) - 1
        summary_obj = ConversationSummary(
            turn_range=(start_turn_id, end_turn_id),
            summary_text=summary["text"],
            token_count=self.counter.count(summary["text"]),
            key_points=summary["key_points"],
            timestamp="2026-06-11T10:00:00Z",
        )
        
        self.summaries.append(summary_obj)
        return summary_obj
    
    def _heuristic_summarize(
        self, turns: list[ConversationTurn]
    ) -> dict[str, Any]:
        """
        启发式摘要生成。
        
        实际场景中应替换为 LLM 调用。此处作为演示。
        """
        user_messages = [t for t in turns if t.role == "user"]
        assistant_messages = [t for t in turns if t.role == "assistant"]
        
        key_points = []
        for t in user_messages:
            # 提取包含问号或关键动词的句子作为关键点
            if "?" in t.content or "?" in t.content:
                key_points.append(f"用户询问: {t.content[:50]}...")
            elif any(kw in t.content for kw in ["要", "想", "需要", "订单", "退货"]):
                key_points.append(f"用户需求: {t.content[:50]}...")
        
        summary_text = f"共 {len(turns)} 轮对话。用户主要关注：" + "; ".join(
            p[:30] for p in key_points[:3]
        )
        
        return {"text": summary_text, "key_points": key_points[:5]}

# 测试摘要压缩
compressor = ConversationCompressor()

# 模拟一段长对话
sample_turns = [
    ConversationTurn(role="user", content="你好，我想咨询退货", turn_id=1),
    ConversationTurn(role="assistant", content="好的，请提供您的订单号", turn_id=2),
    ConversationTurn(role="user", content="ORD-2026-00158", turn_id=3),
    ConversationTurn(role="assistant", content="已查到，您购买的是电子产品，可以7天内退货", turn_id=4),
    ConversationTurn(role="user", content="我想退货，有什么流程", turn_id=5),
    ConversationTurn(role="assistant", content="您需要在线提交退货申请，然后寄回商品", turn_id=6),
]

summary = compressor.compress(sample_turns, start_turn_id=1)

print("=== 对话压缩测试 ===")
original_tokens = sum(t.tokens if t.tokens > 0 else counter.count(t.content) for t in sample_turns)
print(f"原始对话 tokens: ~{original_tokens}")
print(f"摘要 tokens: {summary.token_count}")
if original_tokens > 0:
    compression_ratio = original_tokens / max(summary.token_count, 1)
    print(f"压缩比: {compression_ratio:.1f}x")
print(f"摘要: {summary.summary_text}")
print(f"关键点: {summary.key_points}")

---
## 4. 分层记忆系统 (Hierarchical Memory)

核心思想：不同"距离"的记忆使用不同粒度的表示。

```
短期记忆 (最近3轮)  → 完整保留原始文本
中期记忆 (4-10轮)   → 逐轮摘要
长期记忆 (10+轮)    → 全局总结摘要
```

In [ ]:
# ============================================================
# 分层记忆系统实现
# ============================================================

@dataclass
class MemoryBlock:
    """记忆块：分层记忆的基本单元"""
    level: str  # "short_term" | "mid_term" | "long_term"
    turns: list[ConversationTurn] = field(default_factory=list)
    summary: str = ""
    created_at: int = 0  # turn_id 范围的下界

class HierarchicalMemory:
    """
    分层记忆系统。
    
    What: 三级记忆架构：短期完整保留，中期摘要，长期全局总结。
    Why: 平衡上下文完整性和 token 效率 — 最近的对话保留细节，远的对话保留要点。
    When: 长对话场景（>10 轮），需要访问历史但不需所有细节。
    """
    
    SHORT_TERM_TURNS = 3    # 最近 3 轮 → 短期记忆
    MID_TERM_TURNS = 7      # 之前 7 轮 → 中期记忆  
    # 更早的 → 长期记忆
    
    def __init__(self, model: str = "gpt-4o"):
        self.counter = TokenCounter(model)
        self.compressor = ConversationCompressor(self.counter)
        
        self._all_turns: list[ConversationTurn] = []
        self._system_prompt: str = ""
        
        # 三级记忆块
        self.short_term: MemoryBlock = MemoryBlock(level="short_term")
        self.mid_term: MemoryBlock = MemoryBlock(level="mid_term")
        self.long_term: MemoryBlock = MemoryBlock(level="long_term")
    
    def set_system_prompt(self, prompt: str):
        self._system_prompt = prompt
    
    def add_turn(self, role: str, content: str):
        """添加新的对话轮次并触发记忆整理"""
        turn_id = len(self._all_turns)
        turn = ConversationTurn(
            role=role, content=content,
            tokens=self.counter.count(content),
            turn_id=turn_id
        )
        self._all_turns.append(turn)
        
        # 重新分配记忆层级
        self._rebalance_memory()
    
    def _rebalance_memory(self):
        """
        根据当前轮次总数重新分配三级记忆。
        
        从最新到最旧：短期 → 中期 → 长期
        """
        total = len(self._all_turns)
        
        # 短期：最近 SHORT_TERM_TURNS 轮
        short_start = max(0, total - self.SHORT_TERM_TURNS)
        self.short_term.turns = self._all_turns[short_start:]
        
        # 中期：再往前 MID_TERM_TURNS 轮
        mid_start = max(0, short_start - self.MID_TERM_TURNS)
        mid_turns = self._all_turns[mid_start:short_start]
        self.mid_term.turns = mid_turns
        
        # 如果有中期轮次，生成中期摘要
        if mid_turns:
            summary = self.compressor.compress(mid_turns, mid_start)
            self.mid_term.summary = summary.summary_text
        
        # 长期：剩余的所有轮次
        long_turns = self._all_turns[:mid_start]
        if long_turns:
            summary = self.compressor.compress(long_turns, 0)
            self.long_term.summary = summary.summary_text
        self.long_term.turns = long_turns
    
    def build_context(self) -> str:
        """
        构建要发送给 LLM 的上下文文本。
        
        Returns:
            分层记忆拼接后的完整上下文
        """
        parts = []
        
        # System Prompt
        if self._system_prompt:
            parts.append(self._system_prompt)
        
        # 长期记忆（全局摘要）
        if self.long_term.summary:
            parts.append(f"\n[历史对话摘要]\n{self.long_term.summary}")
        
        # 中期记忆（逐轮摘要）
        if self.mid_term.summary:
            parts.append(f"\n[近期对话摘要]\n{self.mid_term.summary}")
        
        # 短期记忆（完整对话）
        if self.short_term.turns:
            parts.append("\n[当前对话]")
            for turn in self.short_term.turns:
                parts.append(f"[{turn.role}]: {turn.content}")
        
        return "\n".join(parts)
    
    def get_stats(self) -> dict:
        """获取记忆统计信息"""
        return {
            "total_turns": len(self._all_turns),
            "short_term_turns": len(self.short_term.turns),
            "mid_term_turns": len(self.mid_term.turns),
            "long_term_turns": len(self.long_term.turns),
            "context_tokens": self.counter.count(self.build_context()),
        }


# 测试分层记忆系统
memory = HierarchicalMemory()
memory.set_system_prompt("你是一个专业的客户服务助手。")

print("=== 分层记忆演示 ===\n")

# 模拟 15 轮对话
long_conversation = [
    ("user", "你好"),
    ("assistant", "您好！有什么可以帮您的？"),
    ("user", "我想了解你们的会员等级"),
    ("assistant", "我们有三个等级：普通、银卡、金卡..."),
    ("user", "怎么升级到银卡"),
    ("assistant", "消费满2000元自动升级为银卡会员"),
    ("user", "金卡有什么特权"),
    ("assistant", "金卡享受免费配送、专属客服、生日礼品等"),
    ("user", "我现在是什么等级"),
    ("assistant", "您目前是普通会员，已消费1200元"),
    ("user", "噢，那我再消费800就能升级了？"),
    ("assistant", "是的，再消费800元即可升级为银卡会员"),
    ("user", "升级后之前的积分还在吗"),
    ("assistant", "积分不会清零，升级后积分翻倍累计"),
    ("user", "好的谢谢，我先去看看有什么要买的"),
    ("assistant", "好的，有需要随时联系我"),
]

for i, (role, content) in enumerate(long_conversation):
    memory.add_turn(role, content)
    if i in [3, 7, 11, 14]:  # 几个检查点
        stats = memory.get_stats()
        print(f"第 {i+1}/{len(long_conversation)} 轮后:")
        print(f"  短期: {stats['short_term_turns']} 轮 | "
              f"中期: {stats['mid_term_turns']} 轮 | "
              f"长期: {stats['long_term_turns']} 轮")
        print(f"  上下文 tokens: {stats['context_tokens']}")

print(f"\n=== 最终上下文（前400字符）===")
context = memory.build_context()
print(context[:400] + "...")
print(f"\n最终上下文总 tokens: {memory.counter.count(context)}")

---
## 5. 上下文溢出防护 (Context Overflow Prevention)

### 触发规则
```
使用率 < 90%:  正常操作
使用率 ≥ 90%:  触发压缩（压缩中期记忆为长期记忆）
使用率 ≥ 100%: 触发截断（从最旧的消息开始丢弃）
```

In [ ]:
# ============================================================
# 上下文溢出防护系统
# ============================================================

@dataclass
class OverflowAction:
    """溢出防护动作"""
    action: str  # "none" | "compress" | "truncate" | "reject"
    reason: str
    tokens_freed: int = 0

class OverflowGuard:
    """
    上下文溢出防护卫士。
    
    What: 监控上下文使用率，在接近溢出时自动触发压缩或截断。
    Why: 上下文溢出导致静默截断，丢失关键信息，且无错误提示。
    When: 每个 LLM 调用前都应检查。
    """
    
    # 阈值定义
    COMPRESS_THRESHOLD = 0.90   # 90% 触发压缩
    TRUNCATE_THRESHOLD = 1.00   # 100% 触发截断
    
    def __init__(
        self,
        max_tokens: int,
        model: str = "gpt-4o",
    ):
        self.max_tokens = max_tokens
        self.counter = TokenCounter(model)
        self.compressor = ConversationCompressor(self.counter)
    
    def check_and_act(
        self,
        current_messages: list[dict],
        history: list[ConversationTurn],
    ) -> tuple[list[dict], OverflowAction]:
        """
        检查当前消息列表的 token 使用率并执行防护动作。
        
        Args:
            current_messages: 当前准备发送的消息列表
            history: 完整的对话历史（用于压缩）
        Returns:
            (调整后的消息列表, 执行的防护动作)
        """
        total_tokens = self.counter.count_messages(current_messages)
        usage = total_tokens / self.max_tokens
        
        # 正常状态
        if usage < self.COMPRESS_THRESHOLD:
            return current_messages, OverflowAction(
                action="none", reason=f"使用率 {usage:.1%}，正常"
            )
        
        # 触发压缩
        if usage < self.TRUNCATE_THRESHOLD:
            return self._compress_history(current_messages, history)
        
        # 触发截断
        return self._truncate_messages(current_messages)
    
    def _compress_history(
        self,
        messages: list[dict],
        history: list[ConversationTurn],
    ) -> tuple[list[dict], OverflowAction]:
        """
        压缩策略：保留最近 4 条消息的完整内容，
        将之前的消息压缩为摘要并插入为一条 system 消息。
        """
        if len(messages) <= 4 or len(history) <= 6:
            return messages, OverflowAction(
                action="none", reason="消息太少，不值得压缩"
            )
        
        # 找到需要压缩的历史部分
        to_compress = history[:-4]  # 除最近 4 条外的所有历史
        if not to_compress:
            return messages, OverflowAction(
                action="none", reason="无历史可压缩"
            )
        
        summary = self.compressor.compress(to_compress, 0)
        
        # 构建新的消息列表：摘要 → 最近消息
        new_messages = []
        
        # 保留原始 system prompt（如果有）
        if messages and messages[0]["role"] == "system":
            new_messages.append(messages[0])
        
        # 添加历史摘要
        new_messages.append({
            "role": "system",
            "content": f"[对话历史摘要]\n{summary.summary_text}"
        })
        
        # 保留最近的 4 条用户/助手消息
        recent = [m for m in messages[-4:] if m["role"] in ("user", "assistant")]
        new_messages.extend(recent)
        
        old_tokens = self.counter.count_messages(messages)
        new_tokens = self.counter.count_messages(new_messages)
        freed = old_tokens - new_tokens
        
        return new_messages, OverflowAction(
            action="compress",
            reason=f"压缩了 {len(to_compress)} 轮历史",
            tokens_freed=freed,
        )
    
    def _truncate_messages(
        self, messages: list[dict]
    ) -> tuple[list[dict], OverflowAction]:
        """
        截断策略：从最旧的消息开始丢弃（保留 system prompt），
        直到 token 数低于阈值。
        """
        if not messages:
            return messages, OverflowAction(
                action="reject", reason="消息列表为空"
            )
        
        system_msgs = [m for m in messages if m["role"] == "system"]
        other_msgs = [m for m in messages if m["role"] != "system"]
        
        system_tokens = self.counter.count_messages(system_msgs)
        target = int(self.max_tokens * self.COMPRESS_THRESHOLD) - system_tokens
        
        # 从前面开始丢弃非 system 消息
        truncated = []
        current_tokens = 0
        for msg in reversed(other_msgs):  # 从最新往最旧迭代
            msg_tokens = self.counter.count(msg["content"]) + 4
            if current_tokens + msg_tokens <= target:
                truncated.insert(0, msg)
                current_tokens += msg_tokens
            else:
                break
        
        result = system_msgs + truncated
        removed = len(other_msgs) - len(truncated)
        
        if removed == 0:
            return messages, OverflowAction(
                action="none", reason="无法截断（所有消息都需要保留）"
            )
        
        return result, OverflowAction(
            action="truncate",
            reason=f"丢弃了 {removed} 条最早的消息",
            tokens_freed=sum(
                self.counter.count(m["content"]) + 4
                for m in other_msgs[:removed]
            ),
        )


# 测试溢出防护
guard = OverflowGuard(max_tokens=2000, model="gpt-4o")

print("=== 溢出防护演示 ===\n")

# 构建一个接近溢出的消息列表
lorem = "这是一段用于填充token数量的测试文本。" * 50
test_messages = [
    {"role": "system", "content": "你是一个智能助手。"},
    {"role": "user", "content": "第一轮：" + lorem},
    {"role": "assistant", "content": "第一轮回答：" + lorem},
    {"role": "user", "content": "第二轮：" + lorem},
    {"role": "assistant", "content": "第二轮回答：" + lorem},
    {"role": "user", "content": "第三轮：" + lorem},
    {"role": "assistant", "content": "第三轮回答：" + lorem},
]

before_tokens = counter.count_messages(test_messages)
print(f"调整前: {before_tokens} tokens ({before_tokens / guard.max_tokens:.1%})")

# 构建历史
history_turns = [
    ConversationTurn(role=m["role"], content=m["content"])
    for m in test_messages if m["role"] != "system"
]

adjusted, action = guard.check_and_act(test_messages, history_turns)
after_tokens = counter.count_messages(adjusted)
print(f"调整后: {after_tokens} tokens ({after_tokens / guard.max_tokens:.1%})")
print(f"动作: {action.action}")
print(f"原因: {action.reason}")
print(f"释放 token: {action.tokens_freed}")

---
## 本节小结

1. **Token 预算** 是上下文管理的基石，各组件有预设的配额
2. **tiktoken** 提供精确的 token 计数，比字符估算更可靠
3. **滑动窗口** 在长对话中保留最近 N 轮，配合重叠避免断裂
4. **摘要压缩** 用 LLM 将旧对话转为紧凑摘要，压缩比可达 5-20x
5. **分层记忆** 在不同时间粒度上使用不同表示，平衡完整性和效率
6. **溢出防护** 基于阈值自动触发压缩/截断，90% 压缩，100% 截断

### 检查清单
在每次 LLM 调用前：
- [ ] 使用 tiktoken 精确计算 token 使用量
- [ ] 确认 system prompt 不超过窗口的 15%
- [ ] 为输出预留至少 10% 的空间
- [ ] 检查使用率是否超过 90% 的压缩阈值
- [ ] 如果使用率超过 100%，触发截断